In [ ]:
import os
import json
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from rapidfuzz import fuzz, process
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import SparsePCA
from sklearn.model_selection import train_test_split

from scholarlm.utils import get_filenames_in_directory, get_foldernames_in_directory, correct_image_orientation
from scholarlm.judgementlm import tokenize

### Load Data

In [ ]:
main_directory = os.getenv("POND_PATH")
data_directory = os.getenv("POND_DATA_PATH")
pdf_directory = os.getenv("POND_PDF_PATH")
text_directory = os.getenv("POND_TEXT_PATH")
image_directory = os.getenv("POND_IMAGE_PATH")

In [3]:
with open(os.path.join(main_directory, "directory.json"), "r") as f:
    paper_info = json.load(f)

In [4]:
# Directory
with open(os.path.join(main_directory, "directory.json"), "r") as f:
    paper_info = json.load(f)


paper_subset = [
    'physical_and_chemical_limnological',
    'physical-chemical_influences',
    'prairie_wetland',
    'net_heterotrophy',
    'habitat_characteristics',
    'biodiversity_of_constructed',
    'fish_production_in_lakes',
    'long-term_stability',
    'diversity_of_macroinvertebrates',
    'impact_of_macrophytes'
]

paper_info = {k:v for k,v in paper_info.items() if k in paper_subset}


registered_titles = [entry['title'] for entry in paper_info.values()]
registered_titles.sort()
#registered_titles = [registered_titles[i] for i in finished_papers]

# Original dataset
pond_data = pd.read_csv(os.path.join(data_directory, "pond_data.csv"), encoding_errors='ignore')
pond_data = pond_data.loc[pond_data.title.isin(registered_titles)]
pond_df = pond_data.loc[:,['author', 'title', 'pondname', 'location', 'author_term',
            'max_depth_m', 'mean_surfacearea_m2', 'macrophytes_percentcover', 'ph', 'tn_ugpl', 'tp_ugpl', 'chla_ugpl']]
pond_df.columns = ['author', 'title', 'name', 'location', 'ecosystem',
            'max_depth', 'surface_area', 'vegetation_cover', 'ph', 'tn', 'tp', 'chla']

# Split the dataframe's rows so that each measurement is in its own row
pond_df = pond_df.melt(id_vars=['author', 'title', 'name', 'location', 'ecosystem'], 
                       value_vars=['max_depth', 'surface_area', 'vegetation_cover', 'ph', 'tn', 'tp', 'chla'],
                       var_name='measurement', value_name='value')
pond_df = pond_df.dropna(subset=['value'])
pond_df = pond_df.reset_index(drop=True)
n_entries = pond_df.shape[0]

In [61]:
pond_df

,author,title,name,location,ecosystem,measurement,value
0,bortolotti; lauren e.; vinebrooke; rolf d.; st...,prairie wetland communities recover at differe...,hines,saskatchewan,pothole,max_depth,1.1
1,bortolotti; lauren e.; vinebrooke; rolf d.; st...,prairie wetland communities recover at differe...,hood1,saskatchewan,pothole,max_depth,1.4
2,bortolotti; lauren e.; vinebrooke; rolf d.; st...,prairie wetland communities recover at differe...,hood2,saskatchewan,pothole,max_depth,1.7
3,bortolotti; lauren e.; vinebrooke; rolf d.; st...,prairie wetland communities recover at differe...,johanson,saskatchewan,pothole,max_depth,1.0
4,bortolotti; lauren e.; vinebrooke; rolf d.; st...,prairie wetland communities recover at differe...,reinson,saskatchewan,pothole,max_depth,1.1
...,...,...,...,...,...,...,...
1040,peretyatko,impact of macrophytes on phytoplankton in eutr...,malu,brussels; belgium,pond,chla,80.4
1041,peretyatko,impact of macrophytes on phytoplankton in eutr...,wpk2,brussels; belgium,pond,chla,50.6
1042,peretyatko,impact of macrophytes on phytoplankton in eutr...,wpk1,brussels; belgium,pond,chla,41.4
1043,peretyatko,impact of macrophytes on phytoplankton in eutr...,m1k1,brussels; belgium,pond,chla,113.3


In [5]:
def process(
    result_df,
    conversion_table
):
    result_df = result_df.copy()

    # Drop rows without any measurements
    result_df = result_df.dropna(subset=['value'])
    result_df = result_df.reset_index(drop=True)

    # Convert units
    processed_values = [val for val in result_df.loc[:,'value']]
    for row_idx, row in result_df.iterrows():
        measurement_type = row['measurement']
        val = row['value']
        unit = row['units']
        if conversion_table.get(measurement_type) is not None:
            if conversion_table[measurement_type].get(unit) is not None:
                conversion_factor = conversion_table[measurement_type][unit]
                processed_values[row_idx] = val * conversion_factor


    result_df['processed_value'] = processed_values
    result_df = result_df.dropna(subset=['processed_value'])
    result_df = result_df.reset_index(drop=True)

    # Drop all unit columns
    #result_df = result_df.drop(columns=["units"])

    # Drop exact duplicates
    result_df = result_df.drop_duplicates()

    # Reset index
    result_df = result_df.reset_index(drop=True)

    return result_df


conversion_table = {
    'max_depth': {"cm": 0.01, "feet": 0.3048, "km": 1000, "m": 1},
    'surface_area': {"km^2": 1e6, "ha": 1e4, "mi^2": 2.59e6, "m^2": 1},
    'vegetation_cover': {"percent": 0.01, "fraction": 1},
    'tn': {"mg/L": 1000, "µg/L": 1, "μmol/L": 14.01, "ppm": 1000, "ppb": 1},
    'tp': {"mg/L": 1000, "µg/L": 1, "μmol/L": 30.97, "ppm": 1000, "ppb": 1},
    'chla': {"mg/L": 1000, "µg/L": 1},
    'ph': {},
    'latitude': {},
    'longitude': {}
}

In [49]:
import json

def parse_json_string(json_string, default=None):
    """
    Safely parse a JSON string into a dictionary.
    
    Args:
        json_string: The JSON string to parse
        default: The default value to return if parsing fails (default: None)
    
    Returns:
        Dictionary if parsing succeeds, otherwise the default value
    """
    try:
        result = json.loads(json_string.replace('None', 'null'))
        return result
    #except json.JSONDecodeError as e:
    except:
        print('--------------------------------------------')
        print('JSON')
        print(json_string)
        print()
        return default if default is not None else {}

### Parse Elicit Data

In [78]:
elicit_df = pd.read_csv("../data/12_10_25/elicit_extraction2.csv")

In [79]:
extraction_features = [
    'surface_area',
    'max_depth',
    'vegetation_cover',
    'ph',
    'tn',
    'tp',
    'chla'
]
result_dict = []
for i, row in elicit_df.iterrows():
    fname = row['Filename']
    paper_code = fname.replace('.pdf','')
    paper_metadata = paper_info[paper_code]

    for f in extraction_features:
        response = row[f]
        response_dict = parse_json_string(response, default = {'items': []})
        for it in response_dict['items']:
            datapoint = paper_metadata | it | {'measurement': f}
            result_dict.append(datapoint)

--------------------------------------------
JSON
items: [

--------------------------------------------
JSON
items: [

--------------------------------------------
JSON
Not applicable (the paper provides a range of surface areas rather than specific numerical values for individual ecosystems)

--------------------------------------------
JSON
nan

--------------------------------------------
JSON
items: [

--------------------------------------------
JSON
items: [

--------------------------------------------
JSON
items: [

--------------------------------------------
JSON
items: [

--------------------------------------------
JSON
items: [

--------------------------------------------
JSON
items: [

--------------------------------------------
JSON
items: [

--------------------------------------------
JSON
items: [

--------------------------------------------
JSON
items: [

--------------------------------------------
JSON
items: [

--------------------------------------------
JSON

In [80]:
len(result_dict)

269

In [81]:
output_file = "../data/12_10_25/pond_elicit.json"
with open(output_file, "w") as f:
    json.dump(result_dict, f, indent=4, ensure_ascii=False)

In [84]:
with open("../data/12_10_25/pond_elicit_judged.json", "r") as f:
    result_dict = json.load(f)

In [85]:
result_df = pd.DataFrame(result_dict)
result_df = result_df.loc[result_df.title.isin(registered_titles)]
result_df = result_df.reset_index(drop=True)

ignore_measurements = ['latitude', 'longitude'] # Ignoring these for now because they are not in the original dataset

#result_df = result_df.loc[~result_df.measurement.isin(ignore_measurements)]
result_df['value'] = result_df['value'].str.replace(',', '')  # Remove commas from numbers
result_df['value'] = pd.to_numeric(result_df['value'], errors='coerce')
result_df = process(result_df, conversion_table)
result_df.sort_values(by=["title"], inplace=True)
result_df = result_df.reset_index(drop=True)

In [86]:
result_df

,title,author,year,name,date,location,ecosystem,value,units,measurement,judgement,processed_value
0,biodiversity of constructed wetlands for waste...,hsu,2011,HS2-1,null,null,wetland,7.50,percent,vegetation_cover,hallucination,0.075
1,biodiversity of constructed wetlands for waste...,hsu,2011,Hsin-Hai II Constructed Wetland,null,null,constructed wetland,7.43,NaN,ph,hallucination,7.430
2,biodiversity of constructed wetlands for waste...,hsu,2011,Hsin-Hai II Constructed Wetland,null,null,constructed wetland,7.41,NaN,ph,hallucination,7.410
3,biodiversity of constructed wetlands for waste...,hsu,2011,Hsin-Hai II Constructed Wetland,null,null,constructed wetland,7.27,NaN,ph,hallucination,7.270
4,biodiversity of constructed wetlands for waste...,hsu,2011,Hsin-Hai II Constructed Wetland,null,null,constructed wetland,7.20,NaN,ph,hallucination,7.200
...,...,...,...,...,...,...,...,...,...,...,...,...
256,prairie wetland communities recover at differe...,bortolotti; lauren e.; vinebrooke; rolf d.; st...,2016,None,None,"southeastern Saskatchewan, Canada",wetland,7.56,NaN,ph,hallucination,7.560
257,prairie wetland communities recover at differe...,bortolotti; lauren e.; vinebrooke; rolf d.; st...,2016,None,None,"southeastern Saskatchewan, Canada",wetland,7.73,NaN,ph,hallucination,7.730
258,prairie wetland communities recover at differe...,bortolotti; lauren e.; vinebrooke; rolf d.; st...,2016,None,None,"southeastern Saskatchewan, Canada",wetland,246.00,µg/L,tp,hallucination,246.000
259,prairie wetland communities recover at differe...,bortolotti; lauren e.; vinebrooke; rolf d.; st...,2016,None,None,"southeastern Saskatchewan, Canada",wetland,89.00,µg/L,tp,hallucination,89.000


### Match with ground truth

In [87]:
def match_datapoints(ground_truth, extracted):
    edges = []
    edge_weights = []
    for i, row_gt in ground_truth.iterrows():
        for j, row_ex in extracted.iterrows():
            if (
                row_gt['title'] == row_ex['title'] and 
                row_gt['measurement'] == row_ex['measurement'] and 
                np.isclose(row_gt['value'], row_ex['processed_value'], atol=1e-3)
            ):
                name_similarity = fuzz.ratio(row_gt['name'], row_ex['name']) / 100.0
                location_similarity = fuzz.ratio(row_gt['location'], row_ex['location']) / 100.0
                ecosystem_similarity = fuzz.ratio(row_gt['ecosystem'], row_ex['ecosystem']) / 100.0
                weight = (name_similarity + location_similarity + ecosystem_similarity) / 3.0
                edges.append((i, j))
                edge_weights.append(weight)

    print(f"Total edges found: {len(edges)}")
    # Create a bipartite graph and find maximum weight matching
    G = nx.Graph()
    G.add_edges_from([(f"gt_{i}", f"ex_{j}", {'weight': w}) for (i, j), w in zip(edges, edge_weights)])
    matching = nx.algorithms.matching.max_weight_matching(G)
    index_matching = []
    for i, j in matching:
        if i.startswith("gt_"):
            index_matching.append((int(i[3:]), int(j[3:])))
        else:
            index_matching.append((int(j[3:]), int(i[3:])))
    
    return index_matching


def estimate_precision_recall(ground_truth, extracted):
    total_ground_truth = ground_truth.shape[0]
    total_extracted = extracted.shape[0]

    matching = match_datapoints(ground_truth, extracted)

    true_positives = len(matching)
    precision = true_positives / total_extracted if total_extracted > 0 else 0
    recall = true_positives / total_ground_truth if total_ground_truth > 0 else 0

    return recall, precision

In [88]:
matching = match_datapoints(pond_df, result_df)

Total edges found: 319


In [89]:
len(matching)

200

In [90]:
gt_matched = np.array([False] * pond_df.shape[0])
ex_matched = np.array([False] * result_df.shape[0])
for gt_idx, ex_idx in matching:
    gt_matched[gt_idx] = True
    ex_matched[ex_idx] = True

unmatched_gt = np.where(~gt_matched)[0]
unmatched_ex = np.where(~ex_matched)[0]

unmatched_gt_df = pond_df[gt_matched == False]
unmatched_gt_titles = unmatched_gt_df.title.value_counts().index

matched_ex_df = result_df[ex_matched == True]
unmatched_ex_df = result_df[ex_matched == False]
unmatched_ex_titles = unmatched_ex_df.title.value_counts().index

In [41]:
unmatched_gt_df.loc[unmatched_gt_df.measurement == 'surface_area'].title.value_counts()

title
fish production in lakes as a guide for estimating production in proposed reservoirs                                      56
physical-chemical influences on vernal zooplankton community structure in small lakes and wetlands of wisconsin; usa      54
habitat characteristics and odonate diversity in mountain ponds of central italy                                          31
prairie wetland communities recover at different rates following hydrological restoration                                 24
net heterotrophy in small danish lakes: a widespread feature over gradients in trophic status and land cover              18
impact of macrophytes on phytoplankton in eutrophic peri-urban ponds; implications for pond management and restoration    16
biodiversity of constructed wetlands for wastewater treatment                                                             13
Name: count, dtype: int64

In [91]:
unmatched_ex_df.judgement.value_counts()

judgement
hallucination    61
Name: count, dtype: int64

In [92]:
unmatched_ex_df

,title,author,year,name,date,location,ecosystem,value,units,measurement,judgement,processed_value
0,biodiversity of constructed wetlands for waste...,hsu,2011,HS2-1,null,null,wetland,7.50,percent,vegetation_cover,hallucination,0.075
6,biodiversity of constructed wetlands for waste...,hsu,2011,DN5,null,null,wetland,0.00,percent,vegetation_cover,hallucination,0.000
7,biodiversity of constructed wetlands for waste...,hsu,2011,DN3-D,null,null,wetland,6.10,percent,vegetation_cover,hallucination,0.061
9,biodiversity of constructed wetlands for waste...,hsu,2011,DN3-C,null,null,wetland,5.00,percent,vegetation_cover,hallucination,0.050
10,biodiversity of constructed wetlands for waste...,hsu,2011,DN3-A,null,null,wetland,7.60,percent,vegetation_cover,hallucination,0.076
...,...,...,...,...,...,...,...,...,...,...,...,...
248,net heterotrophy in small danish lakes: a wide...,sand-jensen,2009,Slotssø,None,"North Zealand, Denmark",lake,8.10,NaN,ph,hallucination,8.100
254,physical and chemical limnological characteris...,lim et al.,2001,Bathurst Island sites,null,"Bathurst Island, Nunavut, Canada",lakes and ponds,12.70,µg/L,tp,hallucination,12.700
257,prairie wetland communities recover at differe...,bortolotti; lauren e.; vinebrooke; rolf d.; st...,2016,None,None,"southeastern Saskatchewan, Canada",wetland,7.73,NaN,ph,hallucination,7.730
258,prairie wetland communities recover at differe...,bortolotti; lauren e.; vinebrooke; rolf d.; st...,2016,None,None,"southeastern Saskatchewan, Canada",wetland,246.00,µg/L,tp,hallucination,246.000
